In [623]:
import ast
import pandas as pd
import regex as re

In [624]:
df = pd.read_csv("clean/new_cookpad.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 245 entries, 0 to 244
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           245 non-null    int64 
 1   category     245 non-null    object
 2   url          245 non-null    object
 3   title        245 non-null    object
 4   image        245 non-null    object
 5   ingredients  245 non-null    object
 6   steps        245 non-null    object
 7   text         245 non-null    object
 8   steps_text   245 non-null    object
 9   all_text     245 non-null    object
dtypes: int64(1), object(9)
memory usage: 19.3+ KB


In [625]:
new_df = df[['id', 'ingredients', 'steps']]
new_df.head()

,id,ingredients,steps
0,24955519,"['1 butir telur', '1 butir kuning telur', '60 ...","[{'text': 'Siapkan semua bahan. Bahan kulit, b..."
1,24983411,"['500-600 gr Ayam, potong2', '50 gr minyak unt...","[{'text': 'Panaskan minyak, masukkan bawang me..."
2,24975670,['secukupnya Paha ayam dan 2 potong daging aya...,"[{'text': 'Siapkan daging ayam. Pesiang, cuci ..."
3,24983731,"['1 kg ayam negri (dipotong 10)', '1 buah jeru...","[{'text': 'Cuci bersih ayam pada air mengalir,..."
4,24975281,"['2 kg sayap ayam', 'Secukupnya minyak untuk m...","[{'text': 'Cuci bersih sayap ayam, potong jadi..."


In [626]:
import ast
import re
import unicodedata

REMOVE_WORDS = [
    'gr', 'gram', 'ml', 'sdm', 'sdt', 'butir', 'siung', 'lembar', 'potong', 'sec', 'haluskan', 'halus', 'buah', 'sendok', 'cup', 'cm', 'secukupnya', 'sesuai selera', 'lbr', 'untuk menumis', 'diiris serong', 'dipotong kecil', 'kg', 'dipotong', 'bungkus kecil', 'sedikit', 'untuk menggoreng', 'sachet', 'ekor', 'bh', 'se klingking', 'has dalam', 'btg', 'ruas', 'lmbr', 'geprek', 'seiris', 'batang', 'tanpa lemak', 'siap beli','helai', 'tipistipis', 'ikat', 'segenggam', 'g', 'dihaluskan', 'kotak', 'pcs', 'hancurkaniris', 'bulat', 'potongpotong', 'dcc', 'sejumput', 'dll', 'bks', 'irisiris', 'btr', 'genggam', 'makan','lempeng', 'parutpotong', 'lelehkan', 'opsional', 'l', 'tipis', 'digeprek', 'marinasi', 'secukup nya', 'memarkan', 'segitiga', 'diserut', 'optional', 'bungkus', 'serut', 'sisa', 'me', 'kilo',
    'kilogram', 'kecilkecil', 'dicacah', 'giling', 'sckupnya', 'penyedap', 'gelas', 'diparutdiblender', 'centong', 'mentah', 'diblender', 'diparut', 'liter', 'gls', 'st', 'toping', 'parut'
]

REMOVE_PHRASES = [
    'u dibalur ke daging ayam sebelum dipanggang', 'dua boleh ikan yang lain','dan potong daging ayam ukuran kecil','difoto bungkus aslinya aku pakai bungkus agar rasanya lebih mantul','aku pakai produk kecap sedapp','diiris tipis lupa dipakai','saya pakai bon cabe level',' untuk merebus ayam','potong dadu bersihkan dgn jeruk nipis &amp; garam','iris tipis memanjang', 'tambahan saya', 'potong empat bagian','me;tanjung&amp;keriting', 'yang sudah direbusdi presto','saya pilih yang isi', 'buang kulitnya', 'boleh + rawit kalau mau pedas', 'ukuran sedang rebus menit tambahkan daun salam', 'ukuran sedang dadu kecil', 'ambil daunnya saja','ale iris serong cabe ijonya', 'menjadi bagian per', 'biasa bukan asin ukuran cmcm', 'besarcabe keriting', 'buang tinta mulut dan plastik cumi', 'untuk merebus cumi','ukuran sedang bersihkan tintanya biarkan', 'ukuran sedang', 'bersihkan isi perut dan kulit luarnya', 'ambil airnya', 'kocok lepas', 'besar bebas pake tahu apa aja','atau gado instan resep bumbu kacang dibawah kl mau bikin sendiri ya', 'rebus sebentar saja', 'untuk taburan', 'uk besar', 'ukuran kecil iris bulat swm', 'iris tipis korek api','saya pakai ukuran kecil', 'untuk merebus', 'larutkan dengan air', 'aku pakai dancow', 'lelehkan biarkan suhu ruang', 'untuk topping', 'untuk olesan',
    'lumatkan saya pisang marlinkecil', 'jika ingin lebih manis bisa', 'menurutku ini kemanisan jd yg ga suka maniskurangin aja', 'ini sisaan jd seadanya','saya ganti choco chips', 'instan saya skip', 'sy pakai keju oles', 'sy skip', 'me l&#;arome', 'ukuran besar', 'ukuran besar cincang', 'rendam air panas hingga empuk','bisa pisang nangkatanduk', 'beri air utk perekat kulit', 'dingin dikira kira', 'tinggi me cakra kembar', 'sedang me segitiga biru', 'yang sudah', 'dingin bisa juga diganti dengan air dingin','aya pakai pisang saba bandung frozen', 'kuningoranye optional', 'ukuran kecil parut peras saring', 'iris tumis dengan minyak', 'baru mendidih', 'setelah diperas aku skip','setara putih telur', 'untuk pengentalpenyatu isian', 'untuk mencelupkan risoles yg sudah di gulung', 'rendam air panas', 'aku pakai merk finna', 'memanjang dan tipis', 'kecil boleh skip','starter lihat resep', 'sesuaikan selera manis', 'bisa di ganti air hangat', 'untuk oles cetakan', 'di lelegkan', 'aku skip biar anakanak juga bisa pake saus kacang', 'sebanyak yang dibutuhkan','aku pake bawang putih bubuk', 'aku skip karna anakanak nggak suka', 'rebus ambil kaldunya', 'rebus hingga empuk sisihkan jadikan satu dengan kaldu tulang', 'iris memanjang', 'ukuran kecil','masukin di akhir jangan ikut di presto', 'saya pakai telor omega', 'saya labu siam', 'saya skip', 'bila suka', 'atau boleh juga pakai bubuk pala', 'dengan kecap manis dan lada biarkan menit','ambil daun nya', 'selera pedasnya', 'utk menumis', 'kupas cuci bersih', 'boleh skip', 'parut untuk santan santan', 'untuk penyajian', 'penyedap rasa kaldu ayam sesuaikan rasa','resep asli ayam bisa jg pakai ayam kampung', 'bisa diganti jeruk nipis', 'dan bersihkan', 'utuh skip', 'di sampai bijinya keluar', 'sy pake gula pasir', 'buang biji', 'sudah harian','bisa diskip', 'disesuaikan selera', 'takaran sesuaikan', 'sesuai resep yang saya lampirkan', '==&gt;', 'asap lihat resep lodeh ikan lais asap', '&amp;','sy menyesuaikan besarnya daging', 'tambahan kebetulan ada stock paru rebus', 'atau pedasnya', 'atau pedas banyaknya', '&gt; manisnya', 'biar wangi', 'cuci bersih goreng hingga matang sisihkan','supaya warnanya cantik', 'agar hasil lbh gurih', 'satu gigitan', 'masukan saat menumis', 'berlemak aku pakai rib eye', 'bilas dengan air', 'sampai benarbenar matang', 'masukkan setelah bumbu matang','untuk hiasanlalapan', 'utk rebus ayam', 'serong memanjang', 'beli di pajakpasar', 'cuci bersih', 'bisa di kurangi bila tak suka pedas', 'yang serat pinggirnya sudah dikupas','untuk kuah saya banyakin kuahnya karena kuahnya enak', 'kemasan kecil', 'saya pakek yg', 'tinggal cengkeh kayu manis bunga sisir', 'rendam dan rebus', 'yg sdh dibersihkan dan bagian ekornya','aja jangan kebanyakan', 'di geprak daun jeruk', 'besar iris serong', 'bumbu dasar merahresep klik disini', 'ambil air perasannya', 'sesuaikan tingkat kepedasan masingmasing', 'buang tulang daunnya', 'simpulkan', 'jenis apapun', 'jgn banyak', 'sukakalo nggk suka nggk perlu', 'untuk mengoreng', 'kalo suka pake', 'atau bisa pake cuka', 'ambil bagian putihnya','agak banyak kurleb', 'iris serong', 'untuk tumis bumbu', 'dipotek potekin dari batangnya', 'saja jika perlu penyeimbang rasa', 'jika tidak ada pakai pewarna pink muda', 'kocok hingga kaku',
    'atau bisa juga sukade sesuaikan selera', 'saya tidak pakai', 'bisa gula batu gula pasir atau tdk sama sekali', 'note kalau punya bisa diberi cacahan peterseli', 'untuk mengoles', 'untuk compote','untuk compote', 'untuk hiasan irisan', 'bisa diganti daging sapi kalau suka', 'saya pakai yang hot', 'bisa pakai minyak goreng', 'jika punya ortu yang gak punya gigisusah menggigit makanan kenyal silahkan ditambah dengan tepung berasterigu agar lebih mudah untuk dimakan',
    'tikar penggulung kimbap sushi kalo ga ada bisa pakai kertas nasi', 'sy tidak di iris', 'sy ga pakai', 'ops karena saya tidak punya jadi', 'jika tidak ada jamur atau jamur kuping',
    'buat menumis', 'kalian aja isiannya mau apa yaa', 'ayam dicincang', 'iris kecil&quot;', 'lalu dicincang', 'klo gak suka pedas bisa pakai cabe merah secukup', 'saya beli curah d ramayana',
    'utk perasaan', 'hanya untuk daging kambing', '±', 'gelas gelas uk', 'besar dadu', 'tidak pedas cincang kasar', 'gelas uk', 'hingga matang sisihkan', 'daducincang kasar',
    'khusus mpasi', 'tanpa kulit', 'untuk goreng', 'uk sedang', 'boleh ub anchor atau merk lainnya', 'goreng hingga matang', 'atau', 'kupas rebus', 'sekitar', 'pisahkan kuning dan putihnya putih di',
    'matang sesuaikan tekstur dengan kemampuan anak', 'kalau pakai', 'bisa di ganti dengan ayam gilingbakso sosis', 'hangat', 'saya beli jadi', 'pro rendah', 'rebus hancurkan',
    'untuk adonan', 'sedang disangrai', 'di iris panjang', 'topping =', 'boleh mix edam', 'pakai palmia mix margarin', 'menit agar tekstur kering', 'parutsaya masukkan kulkas', 'paruttaburan',
    'perbandingan kuning telur dan teh uht', 'bisa tidak pakai', 'yg telah disangrai dan di cincang kasar', 'pewarna kuning muda minyak goreng', 'sangrai buang kulit lalu',
    'yang telah dikupas', 'dan pewarna kuning', '+', 'mahal', 'blue band', 'bisa pakai pop ice rasa red velvet', 'jika pakai pop ice gula dalam adonan di skip', 'parut taruh dikulkas dulu',
    'panggang cincang agak kasar jangan bubuk dan jangan terlalu kasar juga', 'yg sudah disangrai', 'saya pakai parmigiano', 'keju gouda', 'reggiano', 'icing untuk bedak', 'dari telur',
    'tawar suhu ruang', 'keringkan', 'hasil kurang manis silahkan ditambah', 'mix edamcheddar parut', 'bendorf sangrai', 'saya pakai sapapua timbang setelah disangrai', 'saya pakai susu bubuk',
    'sangrai', 'dan', 'diiris lupa dipakai'
]

def normalize_text(text):
    text = unicodedata.normalize('NFKC', text)
    text = text.replace('\xa0', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.lower().strip()

def clean_ingredients(text):
    ingredients = ast.literal_eval(text)
    cleaned = []
    remove_pattern = r'\b(' + '|'.join(REMOVE_WORDS) + r')\b'
    
    for item in ingredients:
        item = normalize_text(item)
        # hapus angka, tanda baca
        item = re.sub(r'[\d\/\.\-,():]', '', item)
        
        # hapus kata dari daftar REMOVE_WORDS
        item = re.sub(remove_pattern, '', item, flags=re.IGNORECASE)

        # rapikan spasi
        item = re.sub(r'\s+', ' ', item).strip()

        # gunakan .replace() untuk hapus frasa yang tidak diinginkan
        for phrase in REMOVE_PHRASES:
            item = item.replace(phrase, '')
        
        # rapikan spasi
        item = re.sub(r'\s+', ' ', item).strip()
        if len(item) > 2:
            cleaned.append(item)
    
    return ', '.join(cleaned)

new_df['ingredients_clean'] = new_df['ingredients'].apply(clean_ingredients)

C:\Users\User\AppData\Local\Temp\ipykernel_26716\394514528.py:60: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['ingredients_clean'] = new_df['ingredients'].apply(clean_ingredients)


In [627]:
new_df['id'] = new_df['id'].map(lambda x: f"resep-{x}")

C:\Users\User\AppData\Local\Temp\ipykernel_26716\3367630926.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['id'] = new_df['id'].map(lambda x: f"resep-{x}")


In [628]:
new_df = new_df[new_df['ingredients_clean'] != ''].reset_index(drop=True)
new_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 244 entries, 0 to 243
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id                 244 non-null    object
 1   ingredients        244 non-null    object
 2   steps              244 non-null    object
 3   ingredients_clean  244 non-null    object
dtypes: object(4)
memory usage: 7.8+ KB


In [629]:
new_df.head()

,id,ingredients,steps,ingredients_clean
0,resep-24955519,"['1 butir telur', '1 butir kuning telur', '60 ...","[{'text': 'Siapkan semua bahan. Bahan kulit, b...","telur, kuning telur, tepung terigu, maizena ro..."
1,resep-24983411,"['500-600 gr Ayam, potong2', '50 gr minyak unt...","[{'text': 'Panaskan minyak, masukkan bawang me...","ayam, minyak, kaldu ayam, kecap asin, saus tir..."
2,resep-24975670,['secukupnya Paha ayam dan 2 potong daging aya...,"[{'text': 'Siapkan daging ayam. Pesiang, cuci ...","paha ayam daging ayam, jeruk nipis, garam, min..."
3,resep-24983731,"['1 kg ayam negri (dipotong 10)', '1 buah jeru...","[{'text': 'Cuci bersih ayam pada air mengalir,...","ayam negri, jeruk nipis, air, asam jawa, saus ..."
4,resep-24975281,"['2 kg sayap ayam', 'Secukupnya minyak untuk m...","[{'text': 'Cuci bersih sayap ayam, potong jadi...","sayap ayam, minyak, saus tiram, bawang putih b..."


In [630]:
new_df.to_csv("clean/cookpad_final.csv", index=False)